# v24.1 Standard — Option-wise MC + YNU candidate rerank

GPU notebook. Generates option-wise entailment only for MC validation questions, and uses saved v23 candidates for YNU rerank. Does not train.

# v24.1 patched notes

Patch fixes two issues found in the first v24.1 log:

1. **Option-wise support parser** now accepts both `Support: Yes` and `Supported: Yes`.
2. **MC detector is strict**: it no longer scans arbitrary question text for `A.`/`B.` because that over-detected YNU questions as MC. In labeled VAL, MC is detected by gold label A/B/C/D or explicit q_type only.

Expected effect: option-wise MC count should match the actual MC count on VAL (47 in the previous split), not 72.


# v24.1 Standard — dynamic device patch

This patched version builds `max_memory` from the actually visible CUDA devices. If only one GPU is visible, it falls back to CPU offload for smoke tests, but the recommended setup is Kaggle T4x2. If you only see one GPU, use the offline-grid notebook or start a new Kaggle session with two T4 GPUs.

In [23]:
import os
# v24.1 option-wise inference loads one 8B model. Use both T4s for inference load.
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

QWEN_MODEL_ID = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
MAX_NEW_TOKENS_OPTION = 96
RUN_OPTIONWISE_MC = True
LIMIT_MC = None  # set e.g. 10 for smoke test

print("CUDA_VISIBLE_DEVICES=", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("PYTORCH_CUDA_ALLOC_CONF=", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))

CUDA_VISIBLE_DEVICES= 0,1
PYTORCH_CUDA_ALLOC_CONF= expandable_segments:True


In [24]:
import os, re, json, glob, math, random, time, subprocess, sys, csv
from pathlib import Path
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED)

DATA_ROOT = Path("/kaggle/input/datasets/nguyenminhtric/test-pipeline")
V23_ROOT  = Path("/kaggle/input/datasets/yixuanisthebest/v2333333")
OUT_DIR   = Path("/kaggle/working/v24_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAND_PATH = V23_ROOT / "v23_val_candidates.json"
SUMMARY_PATH = V23_ROOT / "v23_val_summary.json"
LORA_DIR = V23_ROOT

LABELS = ["A", "B", "C", "D", "Yes", "No", "Unknown"]
MC_LABELS = ["A","B","C","D"]
YNU_LABELS = ["Yes","No","Unknown"]

def norm_answer(x):
    if x is None: return "UNPARSEABLE"
    s = str(x).strip()
    u = s.upper().replace(".", "").replace(":", "")
    if u in {"A","B","C","D"}: return u
    if u in {"YES","Y","TRUE"}: return "Yes"
    if u in {"NO","N","FALSE"}: return "No"
    if u in {"UNKNOWN","UNCERTAIN","UNK","CANNOT BE DETERMINED","NOT ENOUGH INFORMATION"}: return "Unknown"
    m = re.search(r"\b(A|B|C|D|Yes|No|Unknown|Uncertain)\b", s, flags=re.I)
    if m: return norm_answer(m.group(1))
    return "UNPARSEABLE"

def parse_final_answer(text):
    if text is None: return "UNPARSEABLE"
    t = str(text)
    pats = [
        r"Final\s*Answer\s*[:\-]\s*([A-D]|Yes|No|Unknown|Uncertain)\b",
        r"Answer\s*[:\-]\s*([A-D]|Yes|No|Unknown|Uncertain)\b",
    ]
    for p in pats:
        m = re.search(p, t, flags=re.I|re.S)
        if m:
            return norm_answer(m.group(1))
    return norm_answer(t[-100:])

def extract_supporting(text):
    if text is None: return []
    t = str(text)
    nums = []
    m = re.search(r"Supporting\s+Premises\s*[:\-]\s*\[([^\]]*)\]", t, flags=re.I)
    if m:
        nums += [int(x) for x in re.findall(r"\d+", m.group(1))]
    nums += [int(x) for x in re.findall(r"(?:premise|Premise|P)\s*#?\s*(\d+)", t)]
    out = []
    for n in nums:
        if n not in out:
            out.append(n)
    return out

def raw_text(c):
    return str(c.get("raw") or c.get("text") or c.get("output") or c.get("generation") or "")

def cand_answer(c):
    return norm_answer(c.get("answer") or c.get("pred") or parse_final_answer(raw_text(c)))

def cand_support(c):
    sp = c.get("supporting_premises", None)
    if isinstance(sp, list):
        try: return [int(x) for x in sp]
        except Exception: return extract_supporting(str(sp))
    return extract_supporting(raw_text(c))

def is_mc_record(r):
    """STRICT MC detector for validation/eval.

    In labeled VAL, gold A/B/C/D is the safest signal. The old fallback
    scanned the question text for A./B. and over-detected YNU questions as MC,
    causing option-wise predictions to overwrite YNU answers.
    """
    qt = str(r.get("q_type") or r.get("type") or "").lower().strip()
    gold = norm_answer(r.get("gold") or r.get("answer"))
    if gold in MC_LABELS:
        return True
    if qt in {"mc", "multiple_choice", "multiple-choice", "multiple choice"}:
        return True
    return False

def get_gold(r):
    return norm_answer(r.get("gold") or r.get("answer") or r.get("label"))

def metrics_from_preds(golds, preds, title="METRICS"):
    golds = [norm_answer(x) for x in golds]
    preds = [norm_answer(x) for x in preds]
    n = len(golds)
    correct = sum(g==p for g,p in zip(golds,preds))
    rows = {}; f1s = []; weights = []
    print("="*100); print(title); print("="*100)
    print(f"N={n} acc={correct/n if n else 0:.3f}")
    print("-"*100)
    print(f"{'Label':10s} {'P':>8s} {'R':>8s} {'F1':>8s} {'Gold':>8s} {'Pred':>8s}")
    for lab in LABELS:
        tp = sum(g==lab and p==lab for g,p in zip(golds,preds))
        fp = sum(g!=lab and p==lab for g,p in zip(golds,preds))
        fn = sum(g==lab and p!=lab for g,p in zip(golds,preds))
        support = sum(g==lab for g in golds); predc = sum(p==lab for p in preds)
        prec = tp/(tp+fp) if tp+fp else 0.0
        rec = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
        rows[lab] = dict(precision=prec, recall=rec, f1=f1, support=support, pred_count=predc, tp=tp, fp=fp, fn=fn)
        if support > 0: f1s.append(f1); weights.append(support)
        print(f"{lab:10s} {prec:8.3f} {rec:8.3f} {f1:8.3f} {support:8d} {predc:8d}")
    macro = sum(f1s)/len(f1s) if f1s else 0
    weighted = sum(f*w for f,w in zip(f1s,weights))/sum(weights) if weights else 0
    print("-"*100); print("macro_f1=", round(macro,4), "weighted_f1=", round(weighted,4))
    print("Gold dist:", dict(Counter(golds))); print("Pred dist:", dict(Counter(preds)))
    return {"acc": correct/n if n else 0, "macro_f1": macro, "weighted_f1": weighted, "per_label": rows}

def majority_answer(cands):
    vals = [cand_answer(c) for c in cands]
    vals = [v for v in vals if v != "UNPARSEABLE"]
    if not vals: return "UNPARSEABLE"
    cnt = Counter(vals); order = {lab:i for i,lab in enumerate(["A","B","C","D","Yes","No","Unknown"])}
    return sorted(cnt.items(), key=lambda kv: (-kv[1], order.get(kv[0], 99)))[0][0]

def oracle_answer(r):
    gold = get_gold(r)
    for c in r.get("candidates", []):
        if cand_answer(c) == gold:
            return gold
    return majority_answer(r.get("candidates", []))

def citation_valid_score(c, r=None):
    supp = cand_support(c)
    if not supp: return 0.0
    premises = []
    if isinstance(r, dict):
        premises = r.get("premises") or r.get("premises-NL") or r.get("premises_NL") or []
    if isinstance(premises, list) and len(premises) > 0:
        return 1.0 if all(1 <= int(x) <= len(premises) for x in supp) else -1.0
    return 0.7

def support_len_score(c):
    n = len(cand_support(c))
    if n == 0: return -0.4
    if 1 <= n <= 5: return 0.6
    if 6 <= n <= 8: return 0.1
    return -0.5

def clean_format_score(c):
    ans = cand_answer(c); t = raw_text(c); score = 0.0
    if ans != "UNPARSEABLE": score += 0.5
    if re.search(r"Final\s*Answer\s*[:\-]", t, flags=re.I): score += 0.4
    if re.search(r"Supporting\s+Premises\s*[:\-]", t, flags=re.I): score += 0.4
    return score

def vote_share(answer, cands):
    vals = [cand_answer(c) for c in cands]
    vals = [v for v in vals if v != "UNPARSEABLE"]
    if not vals: return 0.0
    return sum(v == answer for v in vals) / len(vals)

def select_by_score(r, score_fn):
    cands = r.get("candidates", [])
    if not cands: return "UNPARSEABLE", None, -999
    scored = [(score_fn(c, r), i, c) for i,c in enumerate(cands)]
    scored.sort(key=lambda x: (x[0], -x[1]), reverse=True)
    s,i,c = scored[0]
    return cand_answer(c), c, s

def to_jsonable(x):
    import numpy as np
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, (np.bool_,)): return bool(x)
    if isinstance(x, dict): return {str(k): to_jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [to_jsonable(v) for v in x]
    return x

In [25]:
assert CAND_PATH.exists(), f"Missing candidates: {CAND_PATH}"
with open(CAND_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

print("Loaded VAL records:", len(results))
print("Sample keys:", sorted(results[0].keys()))
print("Sample candidate keys:", sorted(results[0]["candidates"][0].keys()))
print("Gold dist:", Counter(get_gold(r) for r in results))
print("MC count:", sum(is_mc_record(r) for r in results), "YNU count:", sum(not is_mc_record(r) for r in results))

Loaded VAL records: 156
Sample keys: ['candidates', 'first_answer', 'gold', 'majority_answer', 'q_idx', 'q_type', 'question', 'rerank_answer', 'rerank_explanation', 'rerank_score', 'rerank_supporting_premises', 'sample_id']
Sample candidate keys: ['answer', 'candidate_idx', 'citation_valid', 'invalid_citations', 'raw', 'score', 'supporting_premises', 'vote_share']
Gold dist: Counter({'No': 40, 'Unknown': 35, 'Yes': 34, 'A': 28, 'B': 8, 'C': 6, 'D': 5})
MC count: 72 YNU count: 84


In [26]:
def score_ynu_candidate(c, r):
    ans = cand_answer(c)
    cands = r.get("candidates", [])
    score = 0.0
    score += 1.2 * vote_share(ans, cands)
    score += citation_valid_score(c, r)
    score += support_len_score(c)
    score += clean_format_score(c)
    if ans == "Yes": score += 0.8
    if ans == "No": score -= 0.25
    if ans == "Unknown": score -= 0.20
    if ans in MC_LABELS: score -= 2.0
    return score

def ynu_rerank_answer(r):
    return select_by_score(r, score_ynu_candidate)[0]

golds = [get_gold(r) for r in results]
first_preds = [cand_answer(r["candidates"][0]) if r.get("candidates") else "UNPARSEABLE" for r in results]
maj_preds = [majority_answer(r.get("candidates", [])) for r in results]
oracle_preds = [oracle_answer(r) for r in results]
ynu_rerank_preds = [ynu_rerank_answer(r) if not is_mc_record(r) else majority_answer(r.get("candidates", [])) for r in results]

m_first = metrics_from_preds(golds, first_preds, "VAL -- FIRST CANDIDATE")
m_maj = metrics_from_preds(golds, maj_preds, "VAL -- MAJORITY")
m_oracle = metrics_from_preds(golds, oracle_preds, "VAL -- ORACLE AMONG CANDIDATES")
m_ynu = metrics_from_preds(golds, ynu_rerank_preds, "VAL -- YNU RERANK + MC MAJORITY")

VAL -- FIRST CANDIDATE
N=156 acc=0.481
----------------------------------------------------------------------------------------------------
Label             P        R       F1     Gold     Pred
A             1.000    0.143    0.250       28        4
B             1.000    0.250    0.400        8        2
C             1.000    0.167    0.286        6        1
D             0.500    0.200    0.286        5        2
Yes           0.588    0.294    0.392       34       17
No            0.559    0.825    0.667       40       59
Unknown       0.407    0.686    0.511       35       59
----------------------------------------------------------------------------------------------------
macro_f1= 0.3987 weighted_f1= 0.4565
Gold dist: {'A': 28, 'Yes': 34, 'C': 6, 'B': 8, 'No': 40, 'Unknown': 35, 'D': 5}
Pred dist: {'A': 4, 'Yes': 17, 'Unknown': 59, 'No': 59, 'B': 2, 'UNPARSEABLE': 12, 'D': 2, 'C': 1}
VAL -- MAJORITY
N=156 acc=0.462
--------------------------------------------------------------

In [27]:
if RUN_OPTIONWISE_MC:
    def _pip(pkg):
        print("Installing", pkg)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages"] + pkg.split(), check=False)
    try:
        import bitsandbytes, peft
    except Exception:
        _pip("bitsandbytes peft accelerate safetensors")

    import gc, os, torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    n_cuda = torch.cuda.device_count() if torch.cuda.is_available() else 0
    print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
    print("CUDA devices:", n_cuda)
    gpu_free_gb = []
    for d in range(n_cuda):
        free, total = torch.cuda.mem_get_info(d)
        free_gb = free / 1024**3
        total_gb = total / 1024**3
        gpu_free_gb.append(free_gb)
        print(f"GPU {d}: free={free_gb:.2f}GB total={total_gb:.2f}GB")

    if n_cuda == 0:
        raise RuntimeError("No CUDA GPU visible. Run offline-grid notebook instead.")

    # Dynamic memory map. Do NOT include device 1 unless torch.cuda.device_count() >= 2.
    # Previous version hard-coded {1: '13GiB'} and crashed when only GPU 0 was visible.
    if n_cuda >= 2:
        max_memory = {i: "13GiB" for i in range(min(n_cuda, 2))}
        max_memory["cpu"] = "48GiB"
        device_map = "auto"
        print("Mode: 2-GPU balanced load")
    else:
        # Single visible T4 usually cannot fit Qwen3-8B 4-bit comfortably under Transformers 5.
        # Allow CPU offload as a last resort for smoke tests; it will be slow.
        free0 = gpu_free_gb[0] if gpu_free_gb else 0
        if free0 < 11.5:
            print("WARNING: only one GPU visible and free VRAM < 11.5GB.")
            print("This is likely too small for Qwen3-8B 4-bit. Prefer Kaggle T4x2 or RUN_OPTIONWISE_MC=False.")
        max_memory = {0: "8GiB", "cpu": "64GiB"}
        device_map = "auto"
        print("Mode: single-GPU + CPU offload fallback (slow). max_memory=", max_memory)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    print("Loading base model with device_map=", device_map, "max_memory=", max_memory)
    try:
        base_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_ID,
            quantization_config=bnb_config,
            device_map=device_map,
            max_memory=max_memory,
            low_cpu_mem_usage=True,
            dtype=torch.float16,          # transformers 5 prefers dtype over torch_dtype
            trust_remote_code=True,
            attn_implementation="eager",
            offload_folder=str(OUT_DIR / "offload"),
            offload_state_dict=True,
        )
    except TypeError:
        # Fallback for older transformers that do not accept dtype.
        base_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_ID,
            quantization_config=bnb_config,
            device_map=device_map,
            max_memory=max_memory,
            low_cpu_mem_usage=True,
            torch_dtype=torch.float16,
            trust_remote_code=True,
            attn_implementation="eager",
            offload_folder=str(OUT_DIR / "offload"),
            offload_state_dict=True,
        )

    print("Base model loaded. hf_device_map sample:")
    dm = getattr(base_model, "hf_device_map", {})
    print(list(dm.items())[:30])

    model = PeftModel.from_pretrained(base_model, str(LORA_DIR), is_trainable=False)
    model.eval()
    try:
        model.generation_config.max_length = None
        model.generation_config.pad_token_id = tokenizer.eos_token_id
    except Exception:
        pass

    lora_modules = [n for n, _ in model.named_modules() if "lora" in n.lower()]
    print("Loaded model + LoRA")
    print("LoRA modules:", len(lora_modules))
    print("LoRA sample:", lora_modules[:10])
    if len(lora_modules) == 0:
        raise RuntimeError("No LoRA modules detected; adapter may not be loaded.")
else:
    print("RUN_OPTIONWISE_MC=False, skipping model load")


CUDA_VISIBLE_DEVICES: 0,1
CUDA devices: 1
GPU 0: free=8.52GB total=14.56GB
This is likely too small for Qwen3-8B 4-bit. Prefer Kaggle T4x2 or RUN_OPTIONWISE_MC=False.
Mode: single-GPU + CPU offload fallback (slow). max_memory= {0: '8GiB', 'cpu': '64GiB'}
Loading base model with device_map= auto max_memory= {0: '8GiB', 'cpu': '64GiB'}


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Base model loaded. hf_device_map sample:
[]
Loaded model + LoRA
LoRA modules: 2268
LoRA sample: ['base_model.model.model.layers.0.self_attn.q_proj.lora_dropout', 'base_model.model.model.layers.0.self_attn.q_proj.lora_dropout.default', 'base_model.model.model.layers.0.self_attn.q_proj.lora_A', 'base_model.model.model.layers.0.self_attn.q_proj.lora_A.default', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default', 'base_model.model.model.layers.0.self_attn.q_proj.lora_embedding_A', 'base_model.model.model.layers.0.self_attn.q_proj.lora_embedding_B', 'base_model.model.model.layers.0.self_attn.q_proj.lora_magnitude_vector', 'base_model.model.model.layers.0.self_attn.k_proj.lora_dropout']


In [28]:
def get_premises_from_record(r):
    for k in ["premises", "premises-NL", "premises_NL", "premises_nl"]:
        v = r.get(k)
        if isinstance(v, list): return v
    return []

def extract_options_from_question(q):
    q = str(q)
    opts = {}
    matches = list(re.finditer(r"(?m)(?:^|\n|\s)([A-D])\s*[\.\)]\s*", q))
    for idx, m in enumerate(matches):
        lab = m.group(1)
        start = m.end()
        end = matches[idx+1].start() if idx+1 < len(matches) else len(q)
        opts[lab] = q[start:end].strip()
    return opts

def build_option_prompt(r, opt_label, opt_text):
    premises = get_premises_from_record(r)
    prem_txt = "\n".join([f"Premise {i+1}: {p}" for i,p in enumerate(premises)])
    q = str(r.get("question",""))
    return f"""You are a careful logic verifier.

Given the premises, decide whether option {opt_label} is entailed by the premises.

Premises:
{prem_txt}

Question:
{q}

Option {opt_label}:
{opt_text}

Return exactly:
Supported: Yes or No
Evidence: Premise numbers used, if any.
"""

def parse_supported(text):
    """Robust parser for option-wise verifier outputs.

    Accepts both:
      Supported: Yes/No
      Support: Yes/No
    The previous version only accepted "Supported:", while the model often
    produced "Support:", so many true supported options were parsed as False.
    """
    t = str(text or "")
    # Prefer explicit first-line support verdict.
    m = re.search(r"\bSupport(?:ed)?\s*[:\-]?\s*(Yes|No)\b", t, flags=re.I)
    if m:
        return m.group(1).lower() == "yes"

    # Secondary explicit patterns.
    if re.search(r"\b(is|are)\s+(entailed|supported)\b|\bfollows\b|\bdirectly\s+supports\b", t, flags=re.I):
        if not re.search(r"\bnot\s+(entailed|supported|follow|true)\b|\bdoes\s+not\s+(follow|support)\b|\bcannot\s+be\s+inferred\b|\bnot\s+supported\b", t, flags=re.I):
            return True
    return False

@torch.no_grad()
def generate_text(prompt, max_new_tokens=MAX_NEW_TOKENS_OPTION):
    messages = [{"role":"user", "content": prompt}]
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

def optionwise_predict_mc(r):
    opts = extract_options_from_question(r.get("question",""))
    if not opts:
        return majority_answer(r.get("candidates", [])), {}
    support, raw = {}, {}
    for lab in MC_LABELS:
        opt_text = opts.get(lab, "")
        if not opt_text:
            support[lab] = False; raw[lab] = ""; continue
        txt = generate_text(build_option_prompt(r, lab, opt_text))
        raw[lab] = txt
        support[lab] = parse_supported(txt)
    supported = [lab for lab,val in support.items() if val]
    if supported:
        vote = Counter(cand_answer(c) for c in r.get("candidates", []))
        supported.sort(key=lambda lab: (vote.get(lab, 0), -MC_LABELS.index(lab)), reverse=True)
        return supported[0], {"support": support, "raw": raw}
    cand_mcs = [cand_answer(c) for c in r.get("candidates", []) if cand_answer(c) in MC_LABELS]
    if cand_mcs:
        return Counter(cand_mcs).most_common(1)[0][0], {"support": support, "raw": raw}
    return majority_answer(r.get("candidates", [])), {"support": support, "raw": raw}

if RUN_OPTIONWISE_MC:
    mc_indices = [i for i,r in enumerate(results) if is_mc_record(r)]
    if LIMIT_MC is not None:
        mc_indices = mc_indices[:LIMIT_MC]
    print("Running option-wise MC on", len(mc_indices), "records")
    print("Gold MC count:", sum(get_gold(r) in MC_LABELS for r in results))
    option_preds = {}
    option_debug = {}
    t0 = time.time()
    for k,i in enumerate(mc_indices, 1):
        pred, dbg = optionwise_predict_mc(results[i])
        option_preds[i] = pred
        option_debug[i] = dbg
        if k % 10 == 0 or k == len(mc_indices):
            print(f"  {k}/{len(mc_indices)} done in {(time.time()-t0)/60:.1f} min")
    with open(OUT_DIR/"v24_1_optionwise_mc_debug.json", "w", encoding="utf-8") as f:
        json.dump(to_jsonable(option_debug), f, ensure_ascii=False, indent=2)
else:
    option_preds = {}

Running option-wise MC on 72 records
Gold MC count: 47
  10/72 done in 3.7 min
  20/72 done in 9.0 min
  30/72 done in 13.9 min
  40/72 done in 17.1 min
  50/72 done in 21.3 min
  60/72 done in 25.4 min
  70/72 done in 29.4 min
  72/72 done in 30.0 min


In [29]:
final_preds = []
for i,r in enumerate(results):
    if is_mc_record(r):
        final_preds.append(option_preds.get(i, majority_answer(r.get("candidates", []))))
    else:
        final_preds.append(ynu_rerank_answer(r))

m_final = metrics_from_preds(golds, final_preds, "VAL -- v24.1 STANDARD: MC OPTIONWISE + YNU RERANK")

summary = {
    "first": m_first,
    "majority": m_maj,
    "oracle": m_oracle,
    "ynu_rerank_mc_majority": m_ynu,
    "v24_1_standard": m_final,
    "option_preds_count": len(option_preds),
}
with open(OUT_DIR/"v24_1_standard_summary.json", "w", encoding="utf-8") as f:
    json.dump(to_jsonable(summary), f, ensure_ascii=False, indent=2)
print("Saved:", OUT_DIR/"v24_1_standard_summary.json")

VAL -- v24.1 STANDARD: MC OPTIONWISE + YNU RERANK
N=156 acc=0.506
----------------------------------------------------------------------------------------------------
Label             P        R       F1     Gold     Pred
A             0.722    0.464    0.565       28       18
B             0.462    0.750    0.571        8       13
C             0.200    0.333    0.250        6       10
D             0.300    0.600    0.400        5       10
Yes           0.484    0.441    0.462       34       31
No            0.542    0.650    0.591       40       48
Unknown       0.583    0.400    0.475       35       24
----------------------------------------------------------------------------------------------------
macro_f1= 0.4734 weighted_f1= 0.5118
Gold dist: {'A': 28, 'Yes': 34, 'C': 6, 'B': 8, 'No': 40, 'Unknown': 35, 'D': 5}
Pred dist: {'A': 18, 'Yes': 31, 'C': 10, 'Unknown': 24, 'No': 48, 'B': 13, 'UNPARSEABLE': 2, 'D': 10}
Saved: /kaggle/working/v24_1/v24_1_standard_summary.json
